[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pradeepvaka/llm-inference-90day/blob/master/notebooks/day20-vllm-scheduling.ipynb)

# Day 20 — vLLM Scheduling: FCFS, Preemption, Chunked Prefill

**Today's big idea:** the scheduler is a network switch's output queue applied to tokens. Every iteration a fixed *token budget* (`max-num-batched-tokens`, default 2048) is split between latency-sensitive decodes and bulk prefills, in strict first-come-first-served order. When KV blocks run out, the *newest* requests are preempted (swap or recompute) so the oldest survive. And a jumbo prefill is *fragmented* into budget-sized chunks piggybacked onto decode iterations — chunked prefill — so one long prompt can never stall the decode pipe.

Below: a faithful discrete-iteration simulator of that scheduler (~80 lines). You will reproduce the 192 ms head-of-line-blocking spike, erase it with chunking, and find the tuning knee for both knobs.


In [ ]:
# No installs needed for the simulation: the lab below is pure Python (math)
# plus matplotlib for the latency traces (preinstalled in Colab).
# The optional vLLM demo in section 7 runs only on a T4 and installs vLLM there,
# guarded by torch.cuda.is_available() -- nothing to do on CPU.
print("CPU lab ready -- no installs needed")

# Expected output:
# CPU lab ready -- no installs needed


## 1. The simulator: a vLLM-style scheduler in ~80 lines

`Req` is one request (prompt length, tokens to generate, prefill/decode progress).
`Scheduler` runs vLLM's per-iteration loop:

1. **Admit** waiting requests, oldest first, while two caps hold: `max_seqs` (live sequences) and the KV block pool. If blocks run out, **preempt the newest running request** — victim goes back to waiting with progress kept (the *swap* model), exactly like vLLM's FCFS victim rule.
2. **Schedule the batch:** 1 decode token per decoding request, then prefill *chunks* (`chunk` tokens each, oldest first) until the token `budget` is spent. `chunk=None` means no chunking: a whole prefill lands in one iteration.
3. **Time the iteration:** `12 + 0.022*prefill_tokens + 0.008*decode_tokens` ms — prefill is compute-bound, decode is memory-bandwidth-bound (your Day 9 numbers, rescaled).

One deliberate simplification: admission reserves the full prompt's KV up front. Real vLLM allocates blocks on demand (Day 19), so it preempts a little later — the dynamics you are about to see are the same, just shifted earlier.


In [ ]:
class Req:
    def __init__(self, rid, arrival, prompt_len, gen_len):
        self.rid, self.arrival = rid, arrival
        self.prompt_len, self.gen_len = prompt_len, gen_len
        self.prefilled = 0      # prompt tokens processed
        self.generated = 0      # decode tokens produced
        self.state = "waiting"  # waiting | running | done
        self.admit_order = None
        self.ttft_iter = None

class Scheduler:
    def __init__(self, budget=2048, chunk=512, max_seqs=256, pool_tokens=16384):
        self.budget, self.chunk, self.max_seqs = budget, chunk, max_seqs
        self.pool = pool_tokens
        self.used = 0
        self.waiting, self.running = [], []
        self.preemptions = 0
        self.admit_seq = 0
        self.iter = 0
        self.trace = []  # (iter_ms, prefill_toks, decode_toks, running_n, waiting_n)

    def kv_tokens(self, r):
        # prompt reserved in full at admission (vLLM reserves max-model-len-ish
        # blocks up front); decodes grow it one token at a time
        return r.prompt_len + r.generated

    def reserve(self, r):
        need = r.prompt_len
        # preempt newest running until the waiting request fits or nothing left to preempt
        while self.used + need > self.pool and self.running:
            victim = max(self.running, key=lambda x: x.admit_order)
            self.running.remove(victim)
            self.used -= self.kv_tokens(victim)
            victim.state = "waiting"
            self.waiting.append(victim)  # swap model: progress kept
            self.preemptions += 1
        return self.used + need <= self.pool

    def admit(self):
        # One pass per iteration over a snapshot (vLLM considers each waiting
        # request once per schedule() call). Victims preempted during this pass
        # rejoin the queue and are reconsidered NEXT iteration -- this bounds
        # the pass and prevents preemption-thrash livelock.
        self.waiting.sort(key=lambda r: r.arrival)
        candidates = [r for r in self.waiting if r.arrival <= self.iter]
        for r in candidates:
            if len(self.running) >= self.max_seqs:
                break
            if self.reserve(r):
                self.waiting.remove(r)
                r.state = "running"
                r.admit_order = self.admit_seq; self.admit_seq += 1
                self.running.append(r)
                self.used += self.kv_tokens(r)
            # else: leave in waiting for a later iteration

    def step(self):
        self.admit()
        remaining = self.budget
        prefill_toks = decode_toks = 0
        # decodes first: 1 token per running request already past prefill
        for r in self.running:
            if r.prefilled >= r.prompt_len and r.generated < r.gen_len and remaining > 0:
                r.generated += 1; self.used += 1; decode_toks += 1; remaining -= 1
                if r.ttft_iter is None:
                    r.ttft_iter = self.iter
        # prefill chunks, oldest admitted first
        # chunk=None means NO chunking: the whole remaining prefill goes in one
        # iteration (the pre-chunked-prefill behavior) -> head-of-line blocking
        for r in sorted(self.running, key=lambda x: x.admit_order):
            if r.prefilled < r.prompt_len:
                if self.chunk is None:
                    c = r.prompt_len - r.prefilled
                else:
                    if remaining <= 0:
                        break
                    c = min(self.chunk, r.prompt_len - r.prefilled, remaining)
                r.prefilled += c  # KV already reserved at admission; no used change
                prefill_toks += c; remaining -= c
        # finish
        for r in list(self.running):
            if r.generated >= r.gen_len:
                r.state = "done"
                self.running.remove(r)
                self.used -= self.kv_tokens(r)
        ms = 12 + 0.022 * prefill_toks + 0.008 * decode_toks
        self.trace.append((ms, prefill_toks, decode_toks, len(self.running), len(self.waiting)))
        self.iter += 1
        return ms

    def run(self, requests, max_iters=400):
        self.all_reqs = requests
        pend = sorted(requests, key=lambda r: r.arrival)
        while self.iter < max_iters:
            while pend and pend[0].arrival <= self.iter:
                self.waiting.append(pend.pop(0))
            self.step()
            if not pend and not self.waiting and not self.running:
                break
        return self

# Expected output: (no output -- class definitions only)


## 2. Burst test: watch preemption keep the server alive

20 requests × 1000-token prompts arrive in the same second. The KV pool holds 16 of the 20 — deliberately tight, so preemption *must* fire. The question: does the server 429, or does it stay live?


In [ ]:
def burst(n=20, prompt=1000, gen=50, arrival=0):
    return [Req(i, arrival, prompt, gen) for i in range(n)]

s = Scheduler(budget=2048, chunk=512, max_seqs=256, pool_tokens=16384).run(burst())
print(f"iterations={s.iter}  preemptions={s.preemptions}")
tt = sorted(r.ttft_iter for r in s.all_reqs)
first_ms = sum(t[0] for t in s.trace[:tt[0]+1])
last_ms = sum(t[0] for t in s.trace[:tt[-1]+1])
print(f"first TTFT: iter {tt[0]} ({first_ms:.0f} ms)   last TTFT: iter {tt[-1]} ({last_ms:.0f} ms)")
print(f"all 20 completed: {all(r.state == 'done' for r in s.all_reqs)}")

# Expected output:
# iterations=97  preemptions=230
# first TTFT: iter 2 (171 ms)   last TTFT: iter 20 (694 ms)
# all 20 completed: True
#
# Read it: a full preemption storm (230 victimizations!) and still zero dropped
# requests. Preemption is backpressure, not failure -- the TCP analogy from the packet.


## 3. Head-of-line blocking: one 8k prompt, no chunking

Steady state: 8 requests (200-token prompts, generating 100 tokens). At iteration 20, one 8192-token prompt arrives. `chunk=None` reproduces the pre-chunked-prefill engine: its whole prefill lands in a single iteration.


In [ ]:
import matplotlib.pyplot as plt

def inject(chunk):
    reqs = [Req(i, 0, 200, 100) for i in range(8)] + [Req(99, 20, 8192, 20)]
    s = Scheduler(budget=2048, chunk=chunk, max_seqs=256, pool_tokens=65536)
    s.run(reqs, max_iters=120)
    return s

s = inject(None)  # no chunking
ms = [t[0] for t in s.trace]
spike_i = max(range(len(ms)), key=lambda i: ms[i])
long = [r for r in s.all_reqs if r.rid == 99][0]
ttft = sum(ms[20:long.ttft_iter+1])
print(f"spike iteration: #{spike_i} at {ms[spike_i]:.0f} ms, prefill_tokens={s.trace[spike_i][1]}")
print(f"long-prompt TTFT: {ttft:.0f} ms; median other iteration: {sorted(ms)[len(ms)//2]:.1f} ms")

plt.figure(figsize=(9, 3.5))
plt.plot(ms)
plt.axvline(20, color="red", linestyle="--", label="8k prompt arrives")
plt.annotate(f"{ms[spike_i]:.0f} ms spike", xy=(spike_i, ms[spike_i]),
             xytext=(spike_i+18, ms[spike_i]+20),
             arrowprops=dict(arrowstyle="->"))
plt.xlabel("iteration"); plt.ylabel("iteration time (ms)")
plt.title("No chunking: one 8192-token prefill stalls every decode")
plt.legend(); plt.tight_layout(); plt.show()

# Expected output:
# spike iteration: #20 at 192 ms, prefill_tokens=8192
# long-prompt TTFT: 204 ms; median other iteration: 12.1 ms
#
# One 192 ms iteration in a sea of 12 ms ones. Every live user feels that as a
# ~192 ms time-per-output-token spike -- the most expensive thing a scheduler
# can do to an interactive fleet.


## 4. Chunked prefill: sixteen smooth iterations

Same injection, `chunk=512`. The 8192-token prompt is fragmented into sixteen 512-token chunks, each piggybacked onto a decode iteration. Watch the spike vanish.


In [ ]:
s = inject(512)
ms = [t[0] for t in s.trace]
long = [r for r in s.all_reqs if r.rid == 99][0]
chunks = sum(1 for t in s.trace[20:] if t[1] == 512)
ttft = sum(ms[20:long.ttft_iter+1])
print(f"max iteration: {max(ms):.1f} ms (the 47 ms is the opening burst's own prefill)")
print(f"prefill-carrying iterations after arrival: {chunks} (expect 16 = 8192/512)")
print(f"long-prompt TTFT: {ttft:.0f} ms; iteration times during prefill: flat ~23.3 ms")

plt.figure(figsize=(9, 3.5))
plt.plot(ms)
plt.axvline(20, color="red", linestyle="--", label="8k prompt arrives")
plt.xlabel("iteration"); plt.ylabel("iteration time (ms)")
plt.title("Chunked prefill (512): 16 smooth iterations, decodes never stall")
plt.legend(); plt.tight_layout(); plt.show()

# Expected output:
# max iteration: 47.2 ms (the 47 ms is the opening burst's own prefill)
# prefill-carrying iterations after arrival: 16 (expect 16 = 8192/512)
# long-prompt TTFT: 385 ms; iteration times during prefill: flat ~23.3 ms
#
# The trade, in numbers: the long prompt's TTFT rises 204 -> 385 ms, and in
# exchange NOBODY else's per-token latency moves. That is the entire argument
# for chunked prefill, and it is why vLLM enables it by default.


## 5. Chunk-size sweep: find the knee

Too small a chunk and the long prompt takes forever; too big and the prefill starves decodes again. Sweep and find the knee — the operating point vLLM's default (512) sits on.


In [ ]:
print(f"{'chunk':>6}  {'long TTFT':>10}  {'TPOT p99':>9}  iters_to_prefill")
for ch in [256, 512, 1024, 2048]:
    s = inject(ch)
    ms = [t[0] for t in s.trace]
    long = [r for r in s.all_reqs if r.rid == 99][0]
    ttft = sum(ms[20:long.ttft_iter+1])
    post = sorted(ms[21:])
    p99 = post[int(0.99 * len(post))]
    print(f"{ch:>6}  {ttft:>8.0f}ms  {p99:>7.1f}ms  {long.ttft_iter - 20}")

# Expected output:
#  chunk   long TTFT   TPOT p99  iters_to_prefill
#    256       578ms     17.7ms  32
#    512       385ms     23.3ms  16
#   1024       289ms     34.6ms   8
#   2048       253ms     56.9ms   5
#
# TTFT falls as the chunk grows; TPOT p99 rises. 512 is the knee: small enough
# that a chunk is ~25% of the budget, large enough that long prompts finish in
# 16 iterations. This is the shape to remember whenever you tune a serving engine.


## 6. `max-num-seqs` sweep: the KV wall

Fix the KV pool, raise the live-sequence cap, and watch throughput stop improving while preemptions explode. The binding constraint is not parallelism — it is Day 19's blocks.


In [ ]:
print(f"{'max_seqs':>8}  {'iters':>5}  {'preemptions':>11}  {'throughput':>10}")
for mseq in [8, 16, 32, 64]:
    s = Scheduler(budget=2048, chunk=512, max_seqs=mseq, pool_tokens=16384).run(burst())
    tot_tok = sum(r.prompt_len + r.gen_len for r in s.all_reqs)
    tot_ms = sum(t[0] for t in s.trace)
    print(f"{mseq:>8}  {s.iter:>5}  {s.preemptions:>11}  {tot_tok/tot_ms*1000:>8.0f} tok/s")

# Expected output:
# max_seqs  iters  preemptions  throughput
#        8    156            0        9052 tok/s
#       16    105            2       12295 tok/s
#       32     97          230       13027 tok/s
#       64     97          230       13027 tok/s
#
# 8 -> 16 buys real throughput. Past 16 the pool is the wall: throughput
# plateaus and every extra admitted sequence just feeds the preemption storm.
# The tuning rule from the packet: push max-num-seqs until the preemption
# counter (vllm:num_preemptions_total) moves, then stop.


## 7. Optional: the real scheduler on a T4

Same A/B you would run against a production vLLM server — `max-num-batched-tokens` 2048 vs 8192 on mixed short/long prompts — using the offline engine on a free T4. Skipped automatically on CPU.


In [ ]:
try:
    import torch
    HAS_GPU = torch.cuda.is_available()
except ImportError:
    HAS_GPU = False
print("cuda available:", HAS_GPU)

if not HAS_GPU:
    print("No GPU -- the CPU simulation above is the full lab;")
    print("switch to a T4 runtime to run the vLLM branch for real.")
else:
    import time, gc
    get_ipython().run_line_magic("pip", "install -q vllm")
    from vllm import LLM, SamplingParams

    prompts = (["Explain TCP backpressure in one sentence."] * 6
               + ["Summarize the plot of the Mahabharata in detail, covering all eighteen parvas: "] * 2)
    sp = SamplingParams(max_tokens=30, temperature=0)

    for budget in [2048, 8192]:
        llm = LLM(model="HuggingFaceTB/SmolLM2-360M", max_model_len=1024,
                  dtype="float16", max_num_batched_tokens=budget,
                  gpu_memory_utilization=0.5)
        t0 = time.time()
        outs = llm.generate(prompts, sp)
        dt = time.time() - t0
        toks = sum(len(o.outputs[0].token_ids) for o in outs)
        print(f"budget={budget}: {toks} tokens in {dt:.1f}s -> {toks/dt:.0f} tok/s")
        del llm; gc.collect(); torch.cuda.empty_cache()

# Expected output (CPU path):
# cuda available: False
# No GPU -- the CPU simulation above is the full lab;
# switch to a T4 runtime to run the vLLM branch for real.
#
# Expected output (T4 path): two throughput lines; 8192 usually shows higher
# tok/s but worse tail latency on the long prompts -- the same trade as section 5.


## What to measure

Fill in from your runs — these are the scheduler's vital signs, and you will compare them across every serving day from here on.

| Metric | Your number |
|---|---|
| Iterations for 8192-token prefill, chunk 512 (sim) | 16 |
| TPOT spike, unchunked 8192 prefill (sim, ms) | ~192 |
| TPOT p99, chunked 512 (sim, ms) | ~23.3 |
| Preemptions under burst of 20 x 1k prompts (sim count) | ~230 |
| Last-request TTFT under burst (sim, ms) | ~694 |
| Live vLLM (optional): tok/s, budget 2048 vs 8192 | |
| Live vLLM (optional): preemptions under gpu-memory-utilization 0.5 burst | |


## Checkpoints

1. **New 8192-token prompt, chunk size 512: how many iterations to finish prefill?**
   16 — 8192 ÷ 512, one chunk per iteration piggybacked on decodes.
2. **Preemption victim selection in vLLM?**
   Newest RUNNING requests first — FCFS preserved for the oldest, which are closest to finishing.
3. **Swap vs recompute for a preempted request: the trade?**
   Swap costs PCIe round-trip time (~20 ms for 256 MiB) and keeps all progress; recompute costs zero traffic but repays the full prefill compute (~67 ms for a 2k-token prompt at 30k tok/s).
4. **Why does chunked prefill raise the long prompt's TTFT while protecting TPOT?**
   The prefill is spread over 16 iterations, so the prompt's first token arrives later — but no iteration is ever dominated by prefill, so every other request's per-token latency stays flat.
5. **You raise `--max-num-seqs` and throughput stops improving while preemptions spike. What's the binding constraint?**
   KV blocks (Day 19): more live sequences than the block pool can hold.


## Tomorrow

**Day 21 — Benchmarking: TTFT, TPOT, Throughput.** Today you learned what the scheduler *does*; tomorrow you build the harness that *measures* it — the load generator, the latency histograms, and the discipline of recording before/after numbers for every optimization from here to Day 90.
